<h3><b style="color:red">IMPORTANT:</b> Restart your session everytime you open this workshop</h3>

In [1]:
%pip install -U evaluate rouge_score rouge chromadb unstructured pypandoc

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━

# Load epub book

In [1]:
# Import libraries
import os
from langchain_community.document_loaders import UnstructuredEPubLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
from uuid import uuid4
from chromadb.utils import embedding_functions

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import CrossEncoder

/tmp/ipykernel_1631/121917837.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredEPubLoader


In [2]:
# TODO: Load document
chunk_size = 1024
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)


In [3]:
# TODO Split document
epub_loader = UnstructuredEPubLoader(file_path="/content/docs/monkeys_paw.epub")

chunks = epub_loader.load_and_split(text_splitter=text_splitter)
print(len(chunks))


27


In [5]:
# TODO Examine chunk
idx = 20
print(chunks[0])

page_content='Cover

THE LADY OF THE BARGE AND OTHER STORIES

By W. W. Jacobs

THE MONKEY’S PAW

I.

Without, the night was cold and wet, but in the small parlour of Laburnam Villa the blinds were drawn and the fire burned brightly. Father and son were at chess, the former, who possessed ideas about the game involving radical changes, putting his king into such sharp and unnecessary perils that it even provoked comment from the white-haired old lady knitting placidly by the fire.

“Hark at the wind,” said Mr. White, who, having seen a fatal mistake after it was too late, was amiably desirous of preventing his son from seeing it.

“I’m listening,” said the latter, grimly surveying the board as he stretched out his hand. “Check.”

“I should hardly think that he’d come to-night,” said his father, with his hand poised over the board.

“Mate,” replied the son.' metadata={'source': '/content/docs/monkeys_paw.epub'}


# Create embeddings

In [6]:
# TODO: Create embedding model
embed_model_name = "BAAI/bge-small-en-v1.5"
#embed_model_name = "all-MiniLM-L6-v2"

# Create the embedding function
embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=embed_model_name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# TODO: Explore embedding model
text = [ 'hello, world', 'big black bug bleeds black blood' ]
emb_text = embed_func(text)

print(len(emb_text))


2


In [9]:
# compare the embedding size for both sentences
print(len(emb_text[0]))
print(len(emb_text[1]))

384
384


In [10]:
print(emb_text[0])

[-3.15818973e-02 -4.86476459e-02  3.21323834e-02 -6.57483339e-02
 -1.12420600e-03  1.14271836e-02 -1.62239338e-03  5.49600422e-02
  4.48704474e-02 -2.09959084e-03  7.87416473e-03 -2.20074486e-02
  3.43550593e-02  6.57046288e-02  2.98711732e-02 -2.77328043e-04
  1.02015038e-03 -3.47685255e-02 -1.21079311e-01 -1.47990771e-02
  9.72586572e-02  3.53694968e-02 -1.68968737e-02 -4.28635962e-02
 -2.48042271e-02  5.63813653e-03  6.80473307e-03  1.35494061e-02
  6.07594289e-03 -9.83635336e-02 -6.45544007e-02 -1.15323914e-02
  3.96090746e-02  2.41095480e-02  4.54738885e-02 -2.10404918e-02
  2.52141263e-02 -1.03886239e-02 -7.94329196e-02  3.64227057e-03
  4.60232235e-02 -5.09503931e-02  1.40664428e-02 -3.41333286e-03
  1.36136143e-02 -4.93411645e-02  1.70672797e-02  5.47222495e-02
 -2.78037898e-02  4.88210673e-04 -5.45994788e-02 -8.51239171e-03
 -1.97877809e-02 -2.24602828e-03  2.84831431e-02  9.09864604e-02
  7.97384903e-02  2.93892017e-03  4.68927547e-02  8.69197026e-03
  1.88648421e-02  1.64884

In [12]:
# TODO: Prepare the chunks for inserting into Chroma
# Extract text for embedding
texts = [ d.page_content for d in chunks]
print(texts)
print(len(texts))


['Cover\n\nTHE LADY OF THE BARGE AND OTHER STORIES\n\nBy W. W. Jacobs\n\nTHE MONKEY’S PAW\n\nI.\n\nWithout, the night was cold and wet, but in the small parlour of Laburnam Villa the blinds were drawn and the fire burned brightly. Father and son were at chess, the former, who possessed ideas about the game involving radical changes, putting his king into such sharp and unnecessary perils that it even provoked comment from the white-haired old lady knitting placidly by the fire.\n\n“Hark at the wind,” said Mr. White, who, having seen a fatal mistake after it was too late, was amiably desirous of preventing his son from seeing it.\n\n“I’m listening,” said the latter, grimly surveying the board as he stretched out his hand. “Check.”\n\n“I should hardly think that he’d come to-night,” said his father, with his hand poised over the board.\n\n“Mate,” replied the son.', '“I should hardly think that he’d come to-night,” said his father, with his hand poised over the board.\n\n“Mate,” replied t

In [14]:
# Create a PK for each text
ids = [ str(uuid4())[:8] for _ in range(len(texts))]
print(ids)
print(len(ids))

['a4f712ac', 'a878819b', 'eaab4263', '4bd3e871', '8640adbc', 'a5f82037', '3ffe7d6e', '49ae7117', '63d8c20b', '86b82271', '8c397f2b', 'f49ad183', '32b1750f', 'addb1e7f', '3afd28df', '28ec5c96', 'e402f1ac', 'ec1d67fe', '8b1749a5', '7d0ed967', '7e634058', '1821ffbe', '4c3b7c58', '351cccac', '7d7afcaf', 'c7683d3b', 'e047e375']
27


In [17]:
# TODO: Create ephemeral Chroma client and save chunks
col_name = 'monkeys_paw'

# Create a db client - memory
ch_client = chromadb.Client()

try:
  ch_client.delete_collection(col_name)
except:
  pass

# Create a collection
monkeypaw_col = ch_client.create_collection(name=col_name, embedding_function=embed_func)

In [18]:
# Insert the docs into the collection
monkeypaw_col.add(
    documents=texts,
    ids=ids
)

In [19]:
# TODO: Print number of documents in collection
print(monkeypaw_col.count())

27


In [31]:
# TODO: Query collection
query = "How many wishes does the monkey's paw grant?"
query = "Tell me about the monkey's paw"

results = monkeypaw_col.query(
    query_texts=query,
    n_results=10
)

for k,v in results.items():
  print(f'key ={k}')
  print(v)

key =ids
[['e047e375', 'f49ad183', '7d0ed967', 'a4f712ac', '4bd3e871', '8640adbc', '63d8c20b', '7d7afcaf', 'e402f1ac', '7e634058']]
key =embeddings
None
key =documents
[['“The bolt,” she cried, loudly. “Come down. I can’t reach it.”\n\nBut her husband was on his hands and knees groping wildly on the floor in search of the paw. If he could only find it before the thing outside got in. A perfect fusillade of knocks reverberated through the house, and he heard the scraping of a chair as his wife put it down in the passage against the door. He heard the creaking of the bolt as it came slowly back, and at the same moment he found the monkey’s paw, and frantically breathed his third and last wish.\n\nThe knocking ceased suddenly, although the echoes of it were still in the house. He heard the chair drawn back, and the door opened. A cold wind rushed up the staircase, and a long loud wail of disappointment and misery from his wife gave him courage to run down to her side, and then to the gate

In [32]:
result_ids = results['ids'][0]
# r = monkeypaw_col.get(ids=result_ids)
# print(r)
# print(len(r['documents']))
for i in result_ids:
  r = monkeypaw_col.get(ids=i)
  print(r)

{'ids': ['e047e375'], 'embeddings': None, 'documents': ['“The bolt,” she cried, loudly. “Come down. I can’t reach it.”\n\nBut her husband was on his hands and knees groping wildly on the floor in search of the paw. If he could only find it before the thing outside got in. A perfect fusillade of knocks reverberated through the house, and he heard the scraping of a chair as his wife put it down in the passage against the door. He heard the creaking of the bolt as it came slowly back, and at the same moment he found the monkey’s paw, and frantically breathed his third and last wish.\n\nThe knocking ceased suddenly, although the echoes of it were still in the house. He heard the chair drawn back, and the door opened. A cold wind rushed up the staircase, and a long loud wail of disappointment and misery from his wife gave him courage to run down to her side, and then to the gate beyond. The street lamp flickering opposite shone on a quiet and deserted road.'], 'uris': None, 'included': ['me

# Cross Encoder
A Cross-Encoder is a model used to determine the semantic similarity, relevance, or relationship between two pieces of text (such as a search query and a document, or a premise and a hypothesis). It embeds 2 pieces of text **together**, allowing full word-by-word cross-attention.

A Bi-Encoder on the other hand embeds 2 pieces of text **independently** into vectors, then calculates similarity (e.g., Cosine Similarity).

In modern RAG application, we combine both bi-encoder and cross-encoder into a two-stage pipeline:

Stage 1: Retrieval (Bi-Encoder): A fast Bi-Encoder searches through millions of documents in a vector database and instantly narrows them down to the top 50 or 100 most likely candidates.

Stage 2: Re-ranking (Cross-Encoder): The slower, highly accurate Cross-Encoder evaluates only those top 50 or 100 candidates side-by-side with the user's query, re-sorting them to ensure the absolute most contextually relevant documents are placed at the very top.

In [30]:
# Use a model with only a single label for reranking
# Takes about approx 6 minutes to load
cross_encode_model = "cross-encoder/ms-marco-MiniLM-L-6-v2"

In [37]:
cross_encoder = CrossEncoder(cross_encode_model)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [41]:
# Rerank the documents
original_order = []
for i in result_ids:
  r = monkeypaw_col.get(ids=i)
  original_order.append(r['documents'][0])

print(len(original_order))

# Rerank the docs and give the top 3
reranked = cross_encoder.rank(query, original_order, top_k=3)

for r in reranked:
  print(r)


10
{'corpus_id': 3, 'score': np.float32(3.3722882)}
{'corpus_id': 6, 'score': np.float32(3.3536568)}
{'corpus_id': 4, 'score': np.float32(1.5714588)}


In [44]:
# Look at the reranked document
print(query)
for r in reranked:
  idx = r['corpus_id']
  print(original_order[idx])
  print('----------------------\n')

Tell me about the monkey's paw
Cover

THE LADY OF THE BARGE AND OTHER STORIES

By W. W. Jacobs

THE MONKEY’S PAW

I.

Without, the night was cold and wet, but in the small parlour of Laburnam Villa the blinds were drawn and the fire burned brightly. Father and son were at chess, the former, who possessed ideas about the game involving radical changes, putting his king into such sharp and unnecessary perils that it even provoked comment from the white-haired old lady knitting placidly by the fire.

“Hark at the wind,” said Mr. White, who, having seen a fatal mistake after it was too late, was amiably desirous of preventing his son from seeing it.

“I’m listening,” said the latter, grimly surveying the board as he stretched out his hand. “Check.”

“I should hardly think that he’d come to-night,” said his father, with his hand poised over the board.

“Mate,” replied the son.
----------------------

“If you must wish,” he said, gruffly, “wish for something sensible.”

Mr. White dropped it 

# Question and Answer LLM
In this exercise you will implement a question and answer LLM for the 'A Christmas Carol' book that you have chunked and saved.

The workflow is as follows:
1. Assume you ask the following question regarding the book eg. `"Who is Scrooge?"`?
2. Query the relevant context from Chroma with the question or facts from the question.
3. Combine the question and the top 5 context return by Chroma into a prompt
4. Use `google/flan-t5-base` to answer the question.

Look through the FLAN templates in [Github](https://github.com/google-research/FLAN/blob/main/flan/templates.py) and select an appropriate template for this workshop.

Do not worry about the accuracy of the result. Focus on implementing the solution. We will discuss the nuances of the solution at the end of the workshop.

Use your RAG workflow to answer the provided questions in `questions_for_rag.txt` file.

In [45]:
# TODO Your code
model = 'google/flan-t5-base'
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForSeq2SeqLM.from_pretrained(model)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [48]:
# TODO Pre-retrieval query rewrite
query = "Who are the three members of the White family?"
query = "Who brings the monkey's paw to the Whites' house?"

rewrite_template = f'{query}\n\nWhat is the sentence that verbalizes this question?'

enc_rewite = tokenizer(rewrite_template, return_tensors='pt')
enc_output = model.generate(**enc_rewite, max_new_tokens=50)

rewrite_query = tokenizer.decode(enc_output[0], skip_special_tokens=True)
print(rewrite_query)

pre_retrieval_query = rewrite_query

The monkey brings the monkey's paw to the Whites' house.


In [66]:
# TODO Pre-retrieval - HyDE
query = "Who brings the monkey's paw to the Whites' house?"
hyde_template = f'Answer the following question. {query}'

enc_hyde = tokenizer(hyde_template, return_tensors='pt')
enc_output = model.generate(**enc_hyde, max_new_tokens=200)

hyde_query = tokenizer.decode(enc_output[0], skip_special_tokens=True)
print(hyde_query)

pre_retrieval_query = hyde_query

john d roosevelt


In [62]:
# TODO Retrieval
top_n = 10
context = monkeypaw_col.query(
    query_texts=pre_retrieval_query,
    n_results=top_n
)
print(context)

{'ids': [['e047e375', '4bd3e871', '63d8c20b', 'eaab4263', '8640adbc', '7d0ed967', 'a878819b', 'f49ad183', '86b82271', 'addb1e7f']], 'embeddings': None, 'documents': [['“The bolt,” she cried, loudly. “Come down. I can’t reach it.”\n\nBut her husband was on his hands and knees groping wildly on the floor in search of the paw. If he could only find it before the thing outside got in. A perfect fusillade of knocks reverberated through the house, and he heard the scraping of a chair as his wife put it down in the passage against the door. He heard the creaking of the bolt as it came slowly back, and at the same moment he found the monkey’s paw, and frantically breathed his third and last wish.\n\nThe knocking ceased suddenly, although the echoes of it were still in the house. He heard the chair drawn back, and the door opened. A cold wind rushed up the staircase, and a long loud wail of disappointment and misery from his wife gave him courage to run down to her side, and then to the gate be

In [57]:
# TODO: Post retrieval - rerank
# Rerank the documents
original_order = []
result_ids = context['ids'][0]
for i in result_ids:
  r = monkeypaw_col.get(ids=i)
  original_order.append(r['documents'][0])

# Rerank the docs and give the top 3
top_n = 2
reranked = cross_encoder.rank(query, original_order, top_k=top_n)

for r in reranked:
  print(r)

{'corpus_id': 2, 'score': np.float32(1.7935379)}
{'corpus_id': 1, 'score': np.float32(0.28400174)}


In [58]:
# TODO: Answer the question
top_n_context = ""
for r in reranked:
  idx = r['corpus_id']
  top_n_context += original_order[idx]
prompt = f"Answer based on context:\n\n{top_n_context}\n\n{query}"

print(prompt)

Answer based on context:

“If you must wish,” he said, gruffly, “wish for something sensible.”

Mr. White dropped it back in his pocket, and placing chairs, motioned his friend to the table. In the business of supper the talisman was partly forgotten, and afterward the three sat listening in an enthralled fashion to a second instalment of the soldier’s adventures in India.

“If the tale about the monkey’s paw is not more truthful than those he has been telling us,” said Herbert, as the door closed behind their guest, just in time for him to catch the last train, “we sha’nt make much out of it.”

“Did you give him anything for it, father?” inquired Mrs. White, regarding her husband closely.

“A trifle,” said he, colouring slightly. “He didn’t want it, but I made him take it. And he pressed me again to throw it away.”

“Likely,” said Herbert, with pretended horror. “Why, we’re going to be rich, and famous and happy. Wish to be an emperor, father, to begin with; then you can’t be henpecke

In [59]:
# TODO: Encode the prompt and evaluate
enc_prompt = tokenizer(prompt, return_tensors='pt')

enc_answer = model.generate(**enc_prompt, max_new_tokens=100)
answer = tokenizer.decode(enc_answer[0], skip_special_tokens=True)

print('------------------------')
print(query)
print(answer)

------------------------
Who brings the monkey's paw to the Whites' house?
Morris


# Optional - Evaluate the answer from you model

Use either Rouge or Bleu to evaluate the answer from your model against those from the text

In [ ]:
# TODO: Evaluate your answer
import evaluate


# Discussion

1. How did your solution perform?
2. Where do you think are the issues?
3. How can you improve it?